# Ngày 24 · Tensor và vòng huấn luyện PyTorch

- Định nghĩa mô hình bằng nn.Module
- Viết vòng huấn luyện chuẩn: zero_grad → forward → loss → backward → step
- Đánh giá đúng cách với model.eval() và torch.no_grad()

**Đọc:**
- Paszke et al. (2019), PyTorch: An Imperative Style, High-Performance Deep Learning Library, NeurIPS 32
- Tài liệu PyTorch, Learn the Basics (Tensors, Datasets & DataLoaders, Build the Neural Network, Automatic Differentiation, Optimization Loop)
- ISLR (bản 2), §10.7 Fitting a Neural Network

In [ ]:
import json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
try:
    import torch
    from torch import nn
    import torch.nn.functional as F
except ImportError as error:
    raise ImportError('Kernel này chưa có PyTorch. Cài bằng Python của Jupyter: python -m pip install -r requirements-py30.txt') from error
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
# load_digits (UCI Optical Recognition of Handwritten Digits): 1797 ảnh 8×8, chia 80/20 phân tầng, seed 0.
_Xd, _yd = load_digits(return_X_y=True)
_itr, _iva = train_test_split(np.arange(len(_yd)), test_size=0.2, random_state=0, stratify=_yd)
X_tr = torch.tensor(_Xd[_itr] / 16, dtype=torch.float32); y_tr = torch.tensor(_yd[_itr])
X_va = torch.tensor(_Xd[_iva] / 16, dtype=torch.float32); y_va = torch.tensor(_yd[_iva])
def _loader(X, y, batch_size, seed):
    return torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=batch_size, shuffle=True,
                                       generator=torch.Generator().manual_seed(seed))
def _train_epoch(model, loader, loss_fn, optimizer):
    model.train(); total, n = 0.0, 0
    for xb, yb in loader:
        optimizer.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); optimizer.step()
        total += loss.item() * len(yb); n += len(yb)
    return total / n
def _accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return float((model(X).argmax(1) == y).float().mean())
import copy
class _Drop(nn.Module):
    def __init__(self):
        super().__init__(); self.net = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.5), nn.Linear(32, 10))
    def forward(self, x): return self.net(x)


## nn.Module, DataLoader và vòng huấn luyện

Gợi ý: MLP: nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(), nn.Linear(hidden, d_out)). make_loader: TensorDataset + DataLoader(shuffle=True, generator=torch.Generator().manual_seed(seed)). evaluate: model.eval() rồi with torch.no_grad().

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_in, hidden, d_out):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

def make_loader(X, y, batch_size, seed):
    raise NotImplementedError

def train_epoch(model, loader, loss_fn, optimizer):
    # Một epoch; trả loss trung bình theo mẫu.
    raise NotImplementedError

def evaluate(model, X, y):
    # Accuracy (float).
    raise NotImplementedError


In [ ]:
def _loader_ok():
    L = make_loader(X_tr, y_tr, 100, 3)
    a = torch.cat([yb for _, yb in L]); b = torch.cat([yb for _, yb in L])
    c = torch.cat([yb for _, yb in make_loader(X_tr, y_tr, 100, 3)])
    sizes = [len(yb) for _, yb in make_loader(X_tr, y_tr, 100, 3)]
    return (sorted(a.tolist()) == sorted(y_tr.tolist()) and not torch.equal(a, b) and torch.equal(a, c)
            and sizes == [100] * 14 + [37])
def _same_step():
    torch.manual_seed(1); m1 = MLP(64, 16, 10); m2 = copy.deepcopy(m1)
    o1, o2 = torch.optim.SGD(m1.parameters(), 0.1), torch.optim.SGD(m2.parameters(), 0.1)
    l1 = train_epoch(m1, make_loader(X_tr, y_tr, 64, 5), nn.CrossEntropyLoss(), o1)
    l2 = _train_epoch(m2, _loader(X_tr, y_tr, 64, 5), nn.CrossEntropyLoss(), o2)
    return np.isclose(l1, l2) and all(torch.allclose(a, b) for a, b in zip(m1.parameters(), m2.parameters()))
def _trained():
    torch.manual_seed(0); m = MLP(64, 64, 10); opt = torch.optim.Adam(m.parameters(), 1e-2); L = make_loader(X_tr, y_tr, 64, 0)
    for _ in range(15): train_epoch(m, L, nn.CrossEntropyLoss(), opt)
    return evaluate(m, X_va, y_va)
def _eval_mode():
    torch.manual_seed(0); m = _Drop()
    a, b = evaluate(m, X_va, y_va), evaluate(m, X_va, y_va)
    return a == b and np.isclose(a, _accuracy(m, X_va, y_va))
_report('d24-loop',
    _check('MLP có đúng số tham số và đầu ra (n, 10)', lambda: ((lambda m: sum(p.numel() for p in m.parameters()) == 4810 and tuple(m(X_va[:5]).shape) == (5, 10))(MLP(64, 64, 10)), '64·64 + 64 + 64·10 + 10 = 4810.')),
    _check('DataLoader trộn theo seed, đủ mẫu, đúng cỡ batch', lambda: (_loader_ok(), '1437 mẫu, batch 100 → 14 batch đủ + 1 batch 37.')),
    _check('Một epoch khớp vòng huấn luyện tham chiếu', lambda: (_same_step(), 'Cùng khởi tạo, cùng thứ tự batch, SGD lr = 0,1.')),
    _check('evaluate dùng model.eval() (Dropout tắt)', lambda: (_eval_mode(), 'Gọi hai lần phải ra cùng kết quả.')),
    _check('Sau 15 epoch Adam: accuracy validation ≥ 0,95', lambda: ((lambda a: (a >= .95, f'Đạt {a:.3f}.'))(_trained())[0], 'Seed 0, lr = 0,01, batch 64.')),
)


## Dự án

Trong notebook cloud, chuyển MLP ngày 23 sang PyTorch (784 → 256 → 10), huấn luyện 5 epoch trên toàn bộ Fashion-MNIST với batch 128 trên GPU nếu có. Ghi thời gian mỗi epoch trên CPU và GPU, loss train và accuracy test.